# Mission 3: KoBERT Multi-label Baseline

119 신고 전화의 전사 텍스트로 9개 증상을 분류하는 KoBERT baseline 실험 노트북이다. 학습 구현은 `train.py`와 `m3` 모듈을 재사용하며, 이 노트북은 설정, 실행, 결과 검증과 시각화를 순서대로 연결한다.

## 0. 환경 및 모듈 준비

현재 실행 위치에서 `mission3_symptom` 디렉터리를 찾고 기존 평가 모듈을 불러온다.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

mission3_dir = Path.cwd()
if not (mission3_dir / "m3").is_dir():
    mission3_dir = mission3_dir / "mission3_symptom"
if not (mission3_dir / "m3").is_dir():
    raise RuntimeError("mission3_symptom 디렉터리에서 실행하거나 repository root에서 실행하세요.")
sys.path.insert(0, str(mission3_dir))

from m3 import TARGET_SYMPTOMS, apply_thresholds, eval_macro_f1, save_thresholds_json
from m3.threshold import find_best_thresholds

plt.rcParams["axes.unicode_minus"] = False
print(f"Mission 3 directory: {mission3_dir.resolve()}")
print(f"Python executable: {Path(sys.executable).resolve()}")

def run_command_live(command):
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
    )
    if process.stdout is None:
        raise RuntimeError("학습 프로세스의 출력을 연결하지 못했습니다.")

    output_buffer = []
    while True:
        character = process.stdout.read(1)
        if character == "":
            if output_buffer:
                print("".join(output_buffer), end="", flush=True)
            break
        output_buffer.append(character)
        if character in {"\r", "\n"}:
            print("".join(output_buffer), end="", flush=True)
            output_buffer.clear()

    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)

## 1. 학습 설정

실험 전에 주로 변경하는 값을 한곳에서 관리한다. CSV는 기본적으로 `mission3_symptom/data` 아래에 있다고 가정하며, 다른 위치에 있다면 `TRAIN_CSV`와 `VAL_CSV`만 수정한다.

In [ ]:
TRAIN_CSV = mission3_dir.parent / "data" / "csv" / "mission3_train.csv"
VAL_CSV = mission3_dir.parent / "data" / "csv" / "mission3_val.csv"
RUN_NAME = "koelectra_base_v3_plain_bce_seed42_val_loss"
OUTPUT_DIR = mission3_dir / "runs" / RUN_NAME

MODEL_NAME = "monologg/koelectra-base-v3-discriminator"
SEED = 42
MAX_LENGTH = 512
TRAIN_BATCH_SIZE = 8
VAL_BATCH_SIZE = 16
GRAD_ACCUM_STEPS = 2
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
EPOCHS = 3
USE_AMP = True
USE_POS_WEIGHT = False

missing_csv = [path for path in (TRAIN_CSV, VAL_CSV) if not path.is_file()]
if missing_csv:
    missing_text = "\n".join(f"- {path.resolve()}" for path in missing_csv)
    raise FileNotFoundError(
        "CSV 파일을 찾을 수 없습니다. TRAIN_CSV와 VAL_CSV를 확인하세요.\n"
        + missing_text
    )

settings = {
    "TRAIN_CSV": TRAIN_CSV.resolve(),
    "VAL_CSV": VAL_CSV.resolve(),
    "RUN_NAME": RUN_NAME,
    "OUTPUT_DIR": OUTPUT_DIR.resolve(),
    "MODEL_NAME": MODEL_NAME,
    "SEED": SEED,
    "MAX_LENGTH": MAX_LENGTH,
    "TRAIN_BATCH_SIZE": TRAIN_BATCH_SIZE,
    "VAL_BATCH_SIZE": VAL_BATCH_SIZE,
    "GRAD_ACCUM_STEPS": GRAD_ACCUM_STEPS,
    "LEARNING_RATE": LEARNING_RATE,
    "WEIGHT_DECAY": WEIGHT_DECAY,
    "WARMUP_RATIO": WARMUP_RATIO,
    "EPOCHS": EPOCHS,
    "USE_AMP": USE_AMP,
    "USE_POS_WEIGHT": USE_POS_WEIGHT,
}
for name, value in settings.items():
    print(f"{name}: {value}")

## 2. Smoke Test

Smoke Test는 모델 성능을 평가하기 위한 실험이 아니다. 전체 학습 전에 **데이터 로딩 → tokenizer → model forward/backward → validation → metric 계산 → 결과 저장**까지 전체 pipeline이 정상 동작하는지만 빠르게 확인하는 테스트다.

일부 Train/Validation sample과 매우 적은 training step만 사용하므로 여기서 나온 loss, Macro F1, 클래스별 F1은 정식 실험 결과로 해석하거나 다른 실험과 비교하면 안 된다. 새로운 환경에서 처음 실행하거나 관련 코드를 변경한 뒤 pipeline의 정상 동작을 확인할 때 실행한다.

아래 셀이 정상 완료된 것을 확인한 뒤 Full Training을 실행한다. Smoke 산출물은 기존 `train.py` 규칙에 따라 정식 run과 분리된 `<RUN_NAME>_smoke` 디렉터리에 저장된다.

In [ ]:
smoke_command = [
    sys.executable,
    "-u",
    str(mission3_dir / "train.py"),
    "--train-csv", str(TRAIN_CSV),
    "--val-csv", str(VAL_CSV),
    "--output-dir", str(OUTPUT_DIR),
    "--model-name-or-path", MODEL_NAME,
    "--seed", str(SEED),
    "--max-length", str(MAX_LENGTH),
    "--train-batch-size", str(TRAIN_BATCH_SIZE),
    "--val-batch-size", str(VAL_BATCH_SIZE),
    "--gradient-accumulation-steps", str(GRAD_ACCUM_STEPS),
    "--learning-rate", str(LEARNING_RATE),
    "--weight-decay", str(WEIGHT_DECAY),
    "--warmup-ratio", str(WARMUP_RATIO),
    "--epochs", str(EPOCHS),
    "--smoke-test",
]
if USE_AMP:
    smoke_command.append("--amp")
if USE_POS_WEIGHT:
    smoke_command.append("--use-pos-weight")

print("Smoke Test 시작")
run_command_live(smoke_command)
print(f"Smoke Test 완료: {OUTPUT_DIR.parent / (OUTPUT_DIR.name + '_smoke')}")

## 3. Full Training

이 단계는 Train 전체 데이터와 Validation 전체 데이터를 사용하는 정식 KoBERT baseline 학습이다. Plain BCE를 사용하고 validation loss가 가장 낮은 checkpoint를 저장한다. Threshold 0.5 Macro F1과 클래스별 F1은 기존처럼 모두 기록한다.

학습 loop를 노트북에 중복 구현하지 않고 1번 셀의 설정으로 기존 `train.py`를 실행한다. `OUTPUT_DIR`이 비어 있지 않으면 기존 정식 run을 보호하기 위해 실행이 중단되므로, 새로운 실험은 `RUN_NAME`을 변경한다.

In [ ]:
full_train_command = [
    sys.executable,
    "-u",
    str(mission3_dir / "train.py"),
    "--train-csv", str(TRAIN_CSV),
    "--val-csv", str(VAL_CSV),
    "--output-dir", str(OUTPUT_DIR),
    "--model-name-or-path", MODEL_NAME,
    "--seed", str(SEED),
    "--max-length", str(MAX_LENGTH),
    "--train-batch-size", str(TRAIN_BATCH_SIZE),
    "--val-batch-size", str(VAL_BATCH_SIZE),
    "--gradient-accumulation-steps", str(GRAD_ACCUM_STEPS),
    "--learning-rate", str(LEARNING_RATE),
    "--weight-decay", str(WEIGHT_DECAY),
    "--warmup-ratio", str(WARMUP_RATIO),
    "--epochs", str(EPOCHS),
]
if USE_AMP:
    full_train_command.append("--amp")
if USE_POS_WEIGHT:
    full_train_command.append("--use-pos-weight")

print("Full Training 시작")
run_command_live(full_train_command)
print(f"Full Training 완료: {OUTPUT_DIR}")

## 4. 정식 Run 산출물 로드

- `run_config.json`: 실제 학습 설정, 데이터 규모, parameter 수, 실행 환경, 토큰 길이 통계
- `history.json`: epoch별 train/validation loss와 threshold 0.5 성능
- `baseline_metrics.json`: best checkpoint의 최종 Validation 결과와 추론 시간
- `val_logits.npy`: best checkpoint의 sigmoid 적용 전 출력
- `val_probs.npy`: logits에 sigmoid를 적용한 확률
- `val_labels.npy`: 동일 순서의 실제 9개 Validation label

In [ ]:
run_dir = OUTPUT_DIR
required_files = [
    "run_config.json",
    "history.json",
    "baseline_metrics.json",
    "val_logits.npy",
    "val_probs.npy",
    "val_labels.npy",
]
missing = [name for name in required_files if not (run_dir / name).is_file()]
if missing:
    raise FileNotFoundError(
        f"정식 run 산출물이 없습니다: {missing}. 먼저 Full Training을 완료하세요."
    )

with open(run_dir / "run_config.json", encoding="utf-8") as file:
    run_config = json.load(file)
with open(run_dir / "history.json", encoding="utf-8") as file:
    history = json.load(file)["epochs"]
with open(run_dir / "baseline_metrics.json", encoding="utf-8") as file:
    saved_metrics = json.load(file)

val_logits = np.load(run_dir / "val_logits.npy")
val_probs = np.load(run_dir / "val_probs.npy")
val_labels = np.load(run_dir / "val_labels.npy")

print(f"Run directory: {run_dir.resolve()}")
print(f"Best epoch: {saved_metrics['best_epoch']}")
print(f"Validation shape: {val_probs.shape}")
print(json.dumps(run_config, ensure_ascii=False, indent=2))

## 5. Threshold 0.5 결과 재검증

저장된 실제 Validation probability에 기존 `apply_thresholds()`와 `eval_macro_f1()`을 다시 적용하여 저장 결과와 일치하는지 확인한다.

In [ ]:
if val_labels.ndim != 2 or val_labels.shape[1] != len(TARGET_SYMPTOMS):
    raise ValueError(f"Validation label 형상이 올바르지 않습니다: {val_labels.shape}")
if val_logits.shape != val_labels.shape or val_probs.shape != val_labels.shape:
    raise ValueError("Validation logits/probabilities/labels 형상이 일치하지 않습니다.")

predictions = apply_thresholds(val_probs, 0.5)
macro_f1, per_class_f1 = eval_macro_f1(
    val_labels, predictions, return_per_class=True
)
if not np.isclose(macro_f1, saved_metrics["val_macro_f1"]):
    raise ValueError("저장된 baseline metric과 재계산 결과가 다릅니다.")

print(f"Macro F1 @ 0.5: {macro_f1:.4f}")
pd.DataFrame({
    "symptom": TARGET_SYMPTOMS,
    "f1": [per_class_f1[symptom] for symptom in TARGET_SYMPTOMS],
})

## 6. 학습 이력 시각화

`history.json`에 저장된 epoch별 loss와 threshold 0.5 Validation Macro F1을 확인한다.

In [ ]:
history_df = pd.DataFrame(history)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history_df["epoch"], history_df["train_loss"], marker="o", label="Train")
axes[0].plot(history_df["epoch"], history_df["val_loss"], marker="o", label="Validation")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[0].grid(alpha=0.3)
axes[1].plot(history_df["epoch"], history_df["val_macro_f1_at_0_5"], marker="o")
axes[1].set_title("Validation Macro F1 @ 0.5")
axes[1].set_xlabel("Epoch")
axes[1].set_ylim(0, 1)
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Threshold 최적화 탐색

Full Training에서 저장한 실제 `val_probs`와 `val_labels`를 기존 `find_best_thresholds()`에 전달하여 클래스별 best threshold를 탐색한다.

팀 확인 전에는 `RUN_THRESHOLD_SEARCH = False`를 유지한다. 이 셀은 탐색 결과를 메모리에만 보관하며 `reports/best_thresholds.json`이나 `reports/comparison.md`를 생성하거나 수정하지 않는다.

In [ ]:
RUN_THRESHOLD_SEARCH = False
threshold_search_completed = False

if RUN_THRESHOLD_SEARCH:
    missing_artifacts = []
    if "run_dir" not in globals():
        missing_artifacts.append("run_dir")
    else:
        for artifact_name in ("val_probs.npy", "val_labels.npy"):
            if not (run_dir / artifact_name).is_file():
                missing_artifacts.append(artifact_name)
    for variable_name in ("val_probs", "val_labels"):
        if variable_name not in globals():
            missing_artifacts.append(variable_name)
    if missing_artifacts:
        raise RuntimeError(
            "먼저 4번에서 정식 run 산출물을 로드하세요: "
            + ", ".join(missing_artifacts)
        )

    (
        best_thresholds,
        threshold_base_macro_f1,
        threshold_optimized_macro_f1,
        threshold_base_per_class_f1,
        threshold_optimized_per_class_f1,
    ) = find_best_thresholds(val_probs, val_labels)
    threshold_search_completed = True

    threshold_table = pd.DataFrame({
        "symptom": TARGET_SYMPTOMS,
        "optimized_threshold": best_thresholds,
    })
    print("Threshold 탐색 완료. reports 파일은 저장하지 않았습니다.")
    print(threshold_table.to_string(index=False))
else:
    print("Threshold 최적화는 실행하지 않았습니다.")

## 8. 최적 Threshold 적용 및 성능 비교

7번에서 탐색한 class-wise threshold를 실제 `val_probs`에 적용하고 threshold 0.5 기준 결과와 비교한다. Threshold 탐색을 다시 수행하지 않으며 기존 `apply_thresholds()`와 `eval_macro_f1()`을 사용해 적용 결과를 재검증한다.

Threshold는 모델 학습이 끝난 뒤 probability에 적용하는 post-processing parameter이므로 이 단계를 위해 Full Training을 다시 실행할 필요가 없다. 탐색을 실행한 경우 해당 run directory에 `optimized_thresholds.json`과 `threshold_metrics.json`을 저장하며 기존 reports 파일은 수정하지 않는다.

In [ ]:
if not globals().get("threshold_search_completed", False):
    print("먼저 7번에서 RUN_THRESHOLD_SEARCH = True로 변경하고 threshold optimization을 실행하세요.")
else:
    optimized_predictions = apply_thresholds(val_probs, best_thresholds)
    optimized_macro_f1, optimized_per_class_f1 = eval_macro_f1(
        val_labels, optimized_predictions, return_per_class=True
    )
    if not np.isclose(optimized_macro_f1, threshold_optimized_macro_f1):
        raise ValueError("Threshold 탐색 결과와 적용 후 재계산 결과가 다릅니다.")

    macro_improvement = optimized_macro_f1 - threshold_base_macro_f1
    print(f"Threshold 0.5 Macro F1: {threshold_base_macro_f1:.4f}")
    print(f"Optimized Threshold Macro F1: {optimized_macro_f1:.4f}")
    print(f"Improvement: {macro_improvement:+.4f}")

    threshold_comparison = pd.DataFrame({
        "symptom": TARGET_SYMPTOMS,
        "threshold": best_thresholds,
        "f1_at_0_5": [
            threshold_base_per_class_f1[symptom]
            for symptom in TARGET_SYMPTOMS
        ],
        "optimized_f1": [
            optimized_per_class_f1[symptom]
            for symptom in TARGET_SYMPTOMS
        ],
    })
    threshold_comparison["f1_delta"] = (
        threshold_comparison["optimized_f1"]
        - threshold_comparison["f1_at_0_5"]
    )
    print(threshold_comparison.to_string(index=False, float_format=lambda value: f"{value:.4f}"))

    if "checkpoint_selection_criterion" not in run_config:
        raise RuntimeError(
            "checkpoint selection criterion이 기록된 새 run에서 실행하세요."
        )
    threshold_metrics = {
        "fixed_threshold": 0.5,
        "macro_f1_at_0.5": float(threshold_base_macro_f1),
        "per_class_f1_at_0.5": {
            symptom: float(threshold_base_per_class_f1[symptom])
            for symptom in TARGET_SYMPTOMS
        },
        "optimized_macro_f1": float(optimized_macro_f1),
        "optimized_per_class_f1": {
            symptom: float(optimized_per_class_f1[symptom])
            for symptom in TARGET_SYMPTOMS
        },
        "optimized_thresholds": {
            symptom: float(best_thresholds[index])
            for index, symptom in enumerate(TARGET_SYMPTOMS)
        },
        "num_val_samples": int(val_labels.shape[0]),
        "checkpoint": saved_metrics["checkpoint"],
        "best_epoch": int(saved_metrics["best_epoch"]),
        "checkpoint_selection_criterion": run_config["checkpoint_selection_criterion"],
    }
    optimized_thresholds_path = save_thresholds_json(
        best_thresholds,
        run_dir / "optimized_thresholds.json",
    )
    threshold_metrics_path = run_dir / "threshold_metrics.json"
    with open(threshold_metrics_path, "w", encoding="utf-8") as file:
        json.dump(threshold_metrics, file, ensure_ascii=False, indent=2)

    print(f"Optimized thresholds 저장: {optimized_thresholds_path}")
    print(f"Threshold metrics 저장: {threshold_metrics_path}")